# Работа с данными бизнеса в ClickHouse

## Задание 1

```sql
/* Для начала продакт-менеджер хочет понять, где сервис пользуется наибольшей популярностью. 
 * Выведите топ-20 городов и регионов России по суммарному количеству 
 * прочитанных и прослушанных часов любого контента с мобильных устройств. 
 * Для каждой из платформ — iOS и Android — добавьте отдельный столбец с длительностью. 
 * Результат должен выглядеть так: город, общая длительность прочитанного и прослушанного 
 * контента, длительность на iOS, длительность на Android. 
 * Значения округлите до целых чисел для лучшей читаемости. 
 * Из выдачи также исключите федеральные округа — оставьте только города и области.
 */
 
select
    usage_geo_id_name,
    round(sumIf(hours_sessions_long, usage_platform_ru in ('Букмейт iOS', 'Букмейт Android'))) total_long,
    round(sumIf(hours_sessions_long, usage_platform_ru in ('Букмейт iOS'))) ios_hours,
    round(sumIf(hours_sessions_long, usage_platform_ru in ('Букмейт Android'))) android_hours
from source_db.audition
where positionCaseInsensitive(usage_geo_id_name, 'федеральный округ') = 0
group by usage_geo_id_name
order by total_long desc
limit 20;
```

| usage_geo_id_name | total_long | ios_hours | android_hours |
|:------------------------------|-----------|-----------|-----------|
| Москва | 26504.0 | 8006.0 | 18499.0 |
| Санкт-Петербург | 12478.0 | 3770.0 | 8708.0 |
| Москва и Московская область | 8211.0 | 2397.0 | 5814.0 |
| Екатеринбург | 4795.0 | 1484.0 | 3311.0 |
| Россия | 4450.0 | 1312.0 | 3137.0 |
| Минск | 4143.0 | 1341.0 | 2803.0 |
| Краснодар | 3978.0 | 1212.0 | 2766.0 |
| Новосибирск | 3808.0 | 1149.0 | 2659.0 |
| Ростов-на-Дону | 3099.0 | 958.0 | 2142.0 |
| Казань | 2985.0 | 1016.0 | 1969.0 |
| Пермь | 2901.0 | 817.0 | 2085.0 |
| Санкт-Петербург и Ленинградская область | 2871.0 | 897.0 | 1974.0 |
| Уфа | 2609.0 | 774.0 | 1834.0 |
| Нижний Новгород | 2590.0 | 698.0 | 1892.0 |
| Челябинск | 2575.0 | 731.0 | 1844.0 |
| Красноярск | 2117.0 | 682.0 | 1435.0 |
| Краснодарский край | 2074.0 | 671.0 | 1403.0 |
| Воронеж | 1833.0 | 548.0 | 1285.0 |
| Тюмень | 1819.0 | 578.0 | 1241.0 |
| Самара | 1800.0 | 555.0 | 1245.0 |

Выводы:     
Максимальное потребление контента происходит в Москве, Санкт-Петербурге и Московской области. Есть общая тенденция, что у Android пользователей количество часов примерно в 2 раза выше, чем на iOS.

## Задание 2

```sql
/* С активными регионами определились, а какой контент самый популярный? 
Получите топ-5 книг по суммарному количеству прочитанных и прослушанных часов 
на мобильных платформах. Также вычислите среднее время чтения и прослушивания в 
зависимости от типа книги: текст или аудио. Результат должен выглядеть так: название книги, 
её автор, суммарное время чтения и прослушивания, среднее время чтения текстовой книги, 
среднее время прослушивания аудиокниги.
В список включайте только те книги, которые используются в обоих форматах. 
Числовые значения округляйте до двух знаков после точки.*/

select
    c.main_content_name as name,
    c.main_author_name as author,
    round(sum(a.hours_sessions_long), 2) as total_long,
    round(avgIf(a.hours_sessions_long, c.main_content_type = 'Book'), 2) as read_avg_hours,
    round(avgIf(a.hours_sessions_long, c.main_content_type = 'Audiobook'), 2) as audio_avg_hours
from source_db.audition a
join source_db.content c on a.main_content_id = c.main_content_id
where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
group by name, author
having hasAll(groupUniqArray(lower(c.main_content_type)), ['book', 'audiobook'])
order by total_long desc
limit 5;
```

| name | author | total_long | read_avg_hours | audio_avg_hours |
|:----------|:-------|------------|-----------------------------|------------------------------------|
| Илон Маск | Уолтер Айзексон | 1006.04 | 0.28 | 0.69 |
| Железное пламя | Ребекка Яррос | 778.71 | 1.74 | 1.88 |
| Убийства и кексики. Детективное агентство «Благотворительный магазин» | Питер Боланд | 539.36 | 0.67 | 1.63 |
| Четвертое крыло | Ребекка Яррос | 499.96 | 1.58 | 1.34 |
| Земля лишних. Трилогия | Андрей Круз | 481.17 | 2.45 | 2.75 |

Выводы:    
    - На первом месте книга `Уолтера Айзексона - Илон Маск`. Пользователи мало тратят времени на прочтение, но слушают в два раза дольше.   
    - Книги от `Ребекки Яррос - Железное пламя`, и `Четвертое крыло` имеют примерно одинаковое время прочтения и прослушивания.      
    - `Питер Боланд - Убийства и кексики. Детективное агентство «Благотворительный магазин»` - прослушиваний в 2,5 раза больше, чем чтения.      
    Стоит отметить так же `Андрей Круз - Земля лишних`. У книги высокое время как прослушивания, так и прочтения. 

## Задание 3

```sql
/* Составьте топ-10 авторов по суммарной длительности чтения их книг на всех платформах, 
включая веб. Для каждого автора добавьте количество уникальных текстовых 
книг (тип контента 'Book' ) и выведите среднюю длительность прослушивания 
их аудиокниг только на мобильных устройствах. 
Исключите авторов, у которых нет аудиокниг.*/

select
    c.main_author_name,
    round(sumIf(a.hours_sessions_long, 
                usage_platform_ru in ('Букмейт iOS', 'Букмейт Android', 'Букмейт Web')),2) total_long,
    uniqExactIf(c.main_content_id, c.main_content_type = 'Book') count_text_book,
    round(avgIf(a.hours_sessions_long, c.main_content_type = 'Audiobook' 
                and usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')), 2) avg_play
from source_db.audition a 
join source_db.content c using(main_content_id)
group by c.main_author_name
having countIf(c.main_content_type = 'Audiobook') > 0
order by total_long desc
limit 10;
```

| main_author_name | total_long | count_text_book | avg_play |
|:----------------------|------------|------------|------------|
| Сергей Лукьяненко | 3192.99 | 54 | 1.75 |
| Александра Лисина | 2236.65 | 71 | 2.27 |
| Анджей Сапковский | 2148.55 | 14 | 1.25 |
| Дарья Донцова | 2088.08 | 163 | 1.9 |
| Борис Акунин | 2034.86 | 52 | 1.48 |
| Фёдор Достоевский | 1734.43 | 29 | 1.14 |
| Артём Каменистый | 1705.69 | 17 | 2.14 |
| Макс Фрай | 1659.39 | 38 | 1.29 |
| Виктор Пелевин | 1619.26 | 30 | 0.95 |
| Андрей Круз | 1514.55 | 14 | 2.04 |

Вывод:    
На первом месте по количеству прослушиваемых часов Сергей Лукьяненко.     
Лидеры по количеству текстовых книг - Дарья Донцова, Александра Лисина. 


## Задание 4

```sql
/* у продакт-менеджера есть предположение, что среди android-пользователей аудиокниги почти 
так же популярны, как тексты. а среди ios-пользователей читателей книг вдвое больше, 
чем слушателей, если считать по суммарной длительности сессии.
проверьте предположение менеджера. для начала выделите три сегмента пользователей:
 - «слушатель» — тот, кто преимущественно пользуется аудиокнигами. 
прослушивание книг составляет 70% и выше от суммарной длительности сессий.
 - «читатель» — преимущественно пользуется текстовыми книгами. чтение книг — от 70%.
 - «оба» — остальные пользователи сервиса.
исключите пользователей, у которых нет сессий ни с книгами, ни с аудиокнигами, 
и посчитайте количество пользователей в каждом из сегментов.
на основе полученных данных проверьте предположение менеджера о том, 
что среди пользователей android примерно одинаковое количество читателей и слушателей, 
а на устройствах ios читателей книг вдвое больше, чем слушателей. 
чтобы определить основную платформу пользователя, учитывайте время её использования. 
например, если пользователь посещал сервис с двух устройств: 
два часа на ios и пять часов на android, то основной платформой такого пользователя будет android.
*/

with user_total as (
    select
        puid,
        sumIf(hours_sessions_long, main_content_type = 'Book') book_hours,
        sumIf(hours_sessions_long, main_content_type = 'Audiobook') audio_hours,
        sumIf(hours_sessions_long, usage_platform_ru in ('Букмейт iOS')) ios_hours,
        sumIf(hours_sessions_long, usage_platform_ru in ('Букмейт Android')) android_hours,
        sum(hours_sessions_long) total_hours
    from source_db.audition a
    join source_db.content c on a.main_content_id = c.main_content_id
    where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
    group by puid
    having book_hours + audio_hours > 0
),
user_main_platform as (
    select
        puid,
        book_hours,
        audio_hours,
        total_hours,
        case
            when ios_hours > android_hours then 'iOS'
            when android_hours > ios_hours then 'Android'
            else 'Android' -- сомнительно, но я так решил
        end main_platform
    from user_total
),
user_category as (
    select
        puid,
        main_platform,
        case
            when book_hours / nullif(total_hours, 0) * 100 >= 70 then 'Читатель'
            when audio_hours / nullif(total_hours, 0) * 100 >= 70 then 'Слушатель'
            else 'Оба'
        end as category
    from user_main_platform
)
select
    main_platform platform,
    category,
    count(distinct puid) count_users
from user_category
where main_platform in ('iOS', 'Android')
group by main_platform, category
order by main_platform, category desc;
```

| platform | category | count_users |
|:-----------|:--------|------------|
| Android | Читатель | 1988 |
| Android | Слушатель | 1850 |
| Android | Оба | 565 |
| iOS | Читатель | 1714 |
| iOS | Слушатель | 1338 |
| iOS | Оба | 364 |

Разница читатель-слушатель Android: 7%     
Разница читатель-слушатель iOS: 22%      

Вывод:    
Отличия есть, на iOS более выраженное, однако гипотеза менеджера о разнице в 2 раза не подтверждается.

## Задание 5

```sql
/* Изучите, существует ли связь между форматом использования приложения (прослушивание или чтение) 
 * и днём недели. Падает ли использование аудиокниг в выходные на всех платформах, включая веб? 
 * Чтобы это узнать, для каждого типа контента посчитайте среднее время его использования в рабочие 
 * и выходные дни и округлите до целого числа. Используя оконные функции, вы также можете пользоваться и комбинаторами.    
 */

select distinct
    main_content_type content_type,
    toDayOfWeek(msk_business_dt_str) day_of_week,
    round(avg(hours_sessions_long*60) over (partition by main_content_type, toDayOfWeek(msk_business_dt_str))) avg_long
from source_db.audition a
join source_db.content c on a.main_content_id = c.main_content_id
where a.usage_platform_ru in ('Букмейт iOS', 'Букмейт Android', 'Букмейт Web')
  and c.main_content_type in ('Book', 'Audiobook')
order by content_type, day_of_week;
```

| content_type | day_of_week | avg_long |
|:----------|--------|---------------------|
| Audiobook | 1 | 78.0 |
| Audiobook | 2 | 76.0 |
| Audiobook | 3 | 78.0 |
| Audiobook | 4 | 77.0 |
| Audiobook | 5 | 76.0 |
| Audiobook | 6 | 79.0 |
| Audiobook | 7 | 82.0 |
| Book | 1 | 48.0 |
| Book | 2 | 46.0 |
| Book | 3 | 48.0 |
| Book | 4 | 47.0 |
| Book | 5 | 46.0 |
| Book | 6 | 50.0 |
| Book | 7 | 52.0 |

Вывод:     
В выходные, на всех платформах чтение и прослушивание книг наоборот, увеличивается. 

## Задание 6

```sql
/* Продакт-менеджер хочет отслеживать обновления приложений на Android и iOS. 
 * У него есть предположение, что больший процент пользователей iOS используют последнюю версию приложения 
 * и в целом чаще его обновляют.
 * Для начала изучите, у какой части пользователей на текущий момент стоят последние версии 
 * приложения на каждой из платформ. Для этого посчитайте последнюю активную версию каждого пользователя 
 * и сравните её с последней версией у каждой платформы. Для каждой платформы выведите процент 
 * пользователей с последней версией приложения и округлите его до двух знаков после точки.
 */
 
with
    user_last_info as (
        select
            puid,
            argMax(app_version, toDateTime(msk_business_dt_str)) user_ver,
            argMax(usage_platform_ru, toDateTime(msk_business_dt_str)) platform
        from source_db.audition
        where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
        group by puid
    ),
    platform_max_version as (
        select
            usage_platform_ru platform,
            max(app_version) platform_ver
        from source_db.audition
        where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
        group by platform
    )
select
    uli.platform,
    round(countIf(user_ver = platform_ver) * 100.0 / count(), 2) pct_latest
from user_last_info uli
join platform_max_version pmv on uli.platform = pmv.platform
group by uli.platform
order by uli.platform;
```

| platform | pct_latest |
|:----------------|----------|
| Букмейт Android | 29.49 |
| Букмейт iOS | 1.92 |

Вывод:      
У пользователей Android, доля пользователей, использующих последнюю актуальную версию приложения, составляет 29,49%.     
Пользователи iOS, с актуальной версией приложения, составляют лишь 1,92%. 

## Задание 7

```sql
/* Теперь продакт-менеджер хочет понять, как часто пользователи обновляют приложение на каждой из 
 * платформ. Фактом обновления считайте изменение версии у каждого пользователя. 
 * Представьте, что любое изменение возможно только в сторону более новой версии. 
 * Проверьте предположение о том, что пользователи iOS чаще обновляют приложение. 
 * Посчитайте метрику update_rate, которая покажет среднюю частоту обновлений на пользователя. 
 * Округлите её до двух знаков после точки.
 */

with sessions as (
    select
        puid,
        usage_platform_ru platform,
        msk_business_dt_str dt,
        app_version
    from source_db.audition
    where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
), ranked as (
    select
        puid,
        platform,
        dt,
        app_version,
        row_number() over (partition by puid, platform order by dt) rank,
        lagInFrame(app_version) over (partition by puid, platform order by dt) prev_version
    from sessions
), updates as (
    select
        puid,
        platform,
        countIf(app_version != prev_version) update_count
    from ranked
    where prev_version is not null
    group by puid, platform
), user_counts as (
    select
        platform,
        count(distinct puid) total_users
    from sessions
    group by platform
)
select
    u.platform,
    round(sum(u.update_count) / uc.total_users, 2) update_rate
from updates u
join user_counts uc on u.platform = uc.platform
group by u.platform, uc.total_users
order by u.platform;
```

| platform | update_rate |
|:----------------|----------|
| Букмейт Android | 3.19 |
| Букмейт iOS | 2.43 |

Вывод:    
Средняя частота обновлений приложения пользователей Android выше, чем у iOS, что опровергает предположение. 

## Задание 8

```sql
/* Новая задача — у коллег есть опасения, что не все книги на тему магии верно размечены 
 * с точки зрения категорий.  Считается, что у книги должно быть не больше 3–4 категорий 
 * с темами. Необходимо найти все книги на магическую тему, которые при этом не входят 
 * в художественную литературу, и проверить, правильно ли они размечены.
 
 * * Начните с подсчёта книг с тегом «Магия». Выведите количество таких книг в каталоге.
 */
 
select countIf(has(published_topic_title_list, 'Магия')) magic_count
from source_db.content
```

| magic_count |
|----------------|
| 46 |

Вывод:    
Количество книг в каталоге, имеющих тег "Магия" - 46 шт.

## Задание 9

```sql
/* Найдите все книги со словом «магия» в названии, для которых не проставлен 
 * тег «Магия». При этом не учитывайте книги с тегом «Художественная литература». 
 * Выведите количество таких книг в каталоге.
 */

select count() AS count
from source_db.content
where main_content_name ILIKE '%магия%'
  and not has(published_topic_title_list, 'Магия')
  and not has(published_topic_title_list, 'Художественная литература');
```


| count |
|----------------|
| 49 |

Вывод:     
Количество книг в каталоге, имеющих в названии слово "Магия", но при этом не имеющие тег "Магия", при условии, что книги не относятся к художественной литературе - 49

## Задание 10

```sql
/* Посчитайте среднее количество категорий у книг с тегом «Магия» и среднее 
 * количество категорий у книг в каталоге в целом. Округлите значения до двух 
 * знаков после точки. Напомним, что коллегам важно, чтобы у каждой книги было 
 * не больше 3–4 категорий. Получится ли не превысить рекомендованного количества?
 */

select 
	round(avgIf(length(published_topic_title_list), has(published_topic_title_list, 'Магия')), 2) avg_magic_count,
	round(avg(length(published_topic_title_list)), 2) all_count
from source_db.content
```

| avg_magic_count | all_count |
|-----------------|-----------|
| 3.22 | 3.77 |

Вывод:     
Среднее количество категорий у книг с тегом "Магия" и в целом, не превышает важного порога в 4 категории.

## Задание 11

```sql
/* Продакт-менеджер выяснил, что в приложении одной из мобильных платформ могла 
 * возникнуть проблема — длина пользовательской сессии (поле hours_sessions_long ) 
 * записывается некорректно, и это происходит как минимум в одной из стран. Чтобы 
 * найти аномалию в данных, используйте такую меру дисперсии как коэффициент вариации. 
 * Напомним его формулу: коэффициент определяется как отношение стандартного отклонения 
 * к среднему. Чем выше этот показатель, тем более подозрительно с точки зрения анализа 
 * распределены данные.
 * Исследуйте коэффициент по странам и мобильным платформам. В какой стране и 
 * на какой платформе видна аномалия в данных? Ограничьте выборку одной страной, 
 * в которой коэффициент вариации для одной из платформ будет наибольшим.
 */

select
    usage_country_name as country,
    usage_platform_ru as platform,
    round(if(avgIf(hours_sessions_long, hours_sessions_long is not null) > 0,
             stddevPopIf(hours_sessions_long, hours_sessions_long is not null) / avgIf(hours_sessions_long, hours_sessions_long is not null),
             0), 2) as cv
from source_db.audition
where usage_platform_ru in ('Букмейт iOS', 'Букмейт Android')
group by country, platform
having avgIf(hours_sessions_long, hours_sessions_long is not null) > 0
order by cv desc
limit 1;
```

| country | platform | cv |
|:-----------|:--------|------------|
| Латвия | Букмейт Android | 7.77 |

Вывод:     
Самый высокий коэффициент вариации в стране латвия, на платформе Android - 7,77. Необходимо проверить корректность записи длины сессий по этой стране 9в рамках платформы).